# 09 — Build Sampling and Decoding Controls

In [ ]:
from pathlib import Path
import math, random, re, json, os, statistics, time, queue, threading
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except Exception as e:
    TfidfVectorizer = None
    cosine_similarity = None
    print('Optional sklearn import failed:', e)

# Make notebooks runnable from the pack root OR from a notebook subdirectory.
def find_pack_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for p in [start] + list(start.parents):
        if (p / 'data' / 'tiny_corpus.txt').exists():
            return p
    # Fallback for the sandbox path used to create this pack.
    sandbox = Path('/mnt/data/advanced_llm_systems_notebook_pack')
    if (sandbox / 'data' / 'tiny_corpus.txt').exists():
        return sandbox
    return start

ROOT = find_pack_root()
DATA = ROOT / 'data'
print('Pack root:', ROOT)
print('PyTorch:', torch.__version__)
torch.set_num_threads(1)
torch.manual_seed(7)
random.seed(7)
np.random.seed(7)

def basic_tokenize(text: str):
    return re.findall(r"[a-zA-Z]+|\d+|[^\w\s]", text.lower())

def count_params(model):
    return sum(p.numel() for p in model.parameters())

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B,T,C = x.shape
        q,k,v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k = k.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        v = v.view(B,T,self.n_heads,self.d_head).transpose(1,2)
        att = q @ k.transpose(-2,-1) / math.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T,T,device=x.device)).bool()
        att = att.masked_fill(~mask[None,None,:,:], float('-inf'))
        w = torch.softmax(att, dim=-1)
        y = self.dropout(w) @ v
        y = y.transpose(1,2).contiguous().view(B,T,C)
        return self.proj(y)

class TransformerBlock(nn.Module):
    def __init__(self, d_model=64, n_heads=4, mlp_ratio=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio*d_model), nn.GELU(),
            nn.Linear(mlp_ratio*d_model, d_model), nn.Dropout(dropout)
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

# Shared toy RAG utilities for notebooks that need retrieval.
def load_rag_docs():
    path = DATA / 'rag_docs.jsonl'
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]

docs = load_rag_docs()
_retriever_state = {}
def build_retriever(docs_in=None):
    docs_in = docs if docs_in is None else docs_in
    if not docs_in or TfidfVectorizer is None:
        return None, None
    vectorizer = TfidfVectorizer(stop_words='english')
    X = vectorizer.fit_transform([d['title'] + ' ' + d['text'] for d in docs_in])
    return vectorizer, X

_default_vectorizer, _default_X = build_retriever(docs)
def retrieve(query, k=2):
    if _default_vectorizer is None or _default_X is None:
        return []
    q = _default_vectorizer.transform([query])
    sims = cosine_similarity(q, _default_X)[0]
    idx = sims.argsort()[::-1][:k]
    return [{**docs[i], 'score': float(sims[i])} for i in idx]

# Shared eval helpers.
def exact_contains(output, expected):
    return str(expected).lower() in str(output).lower()

def valid_json_with_key(output, key):
    try:
        return key in json.loads(output)
    except Exception:
        return False

def run_eval(cases, system_fn):
    rows = []
    for c in cases:
        out = system_fn(c['input'])
        if 'expected' in c:
            passed = exact_contains(out, c['expected'])
        elif 'expected_key' in c:
            passed = valid_json_with_key(out, c['expected_key'])
        else:
            passed = False
        rows.append({**c, 'output': out, 'passed': passed})
    return pd.DataFrame(rows)


## Why this matters

Training gives you a probability distribution. Decoding turns that distribution into text.

Sampling controls are product controls:

- customer-support replies need lower randomness;
- brainstorming can use higher randomness;
- regulated outputs may require constrained decoding or validation.

In [ ]:
vocab = ['<eos>', 'the', 'product', 'refund', 'policy', 'is', 'clear', 'unclear', '.', 'fast', 'slow']
base_logits = torch.tensor([0.0, 2.2, 1.5, 1.4, 1.3, 2.0, 0.9, 0.4, 1.8, 0.3, 0.2])

def apply_temperature(logits, temperature=1.0):
    return logits / max(temperature, 1e-6)

def top_k_filter(logits, k):
    if k is None or k >= logits.numel(): return logits
    values, _ = torch.topk(logits, k)
    cutoff = values[-1]
    return logits.masked_fill(logits < cutoff, float('-inf'))

def top_p_filter(logits, p=0.9):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = torch.softmax(sorted_logits, dim=-1)
    cum = torch.cumsum(probs, dim=-1)
    remove = cum > p
    remove[1:] = remove[:-1].clone()
    remove[0] = False
    filtered = logits.clone()
    filtered[sorted_idx[remove]] = float('-inf')
    return filtered

def sample_token(logits, temperature=1.0, top_k=None, top_p=None):
    z = apply_temperature(logits, temperature)
    z = top_k_filter(z, top_k)
    if top_p is not None:
        z = top_p_filter(z, top_p)
    probs = torch.softmax(z, dim=-1)
    idx = torch.multinomial(probs, 1).item()
    return idx, probs


In [ ]:
for cfg in [
    {'temperature':0.3}, {'temperature':1.0}, {'temperature':1.5},
    {'temperature':1.0, 'top_k':3}, {'temperature':1.0, 'top_p':0.8}
]:
    counts = Counter()
    for _ in range(1000):
        idx, _ = sample_token(base_logits, **cfg)
        counts[vocab[idx]] += 1
    print(cfg, counts.most_common(5))


## Repetition penalty

Repetition penalty reduces logits for tokens already generated. This is a simple version; production systems use more careful variants.

In [ ]:
def repetition_penalty(logits, generated_ids, penalty=1.2):
    logits = logits.clone()
    for i in set(generated_ids):
        logits[i] = logits[i] / penalty if logits[i] > 0 else logits[i] * penalty
    return logits

generated = [vocab.index('the'), vocab.index('product'), vocab.index('is')]
print('before:', dict(zip(vocab, base_logits.tolist())))
print('after :', dict(zip(vocab, repetition_penalty(base_logits, generated).round(decimals=2).tolist())))


## Beam search

Beam search keeps the best partial sequences. It is common in translation and structured generation, but can sound bland for open-ended chat.

In [ ]:
def toy_next_logits(prefix):
    # A fake next-token model for demonstration.
    logits = base_logits.clone()
    if prefix and prefix[-1] == vocab.index('.'):
        logits[0] += 3.0
    if len(prefix) > 6:
        logits[vocab.index('.')] += 2.0
    return logits

def beam_search(width=3, max_len=8):
    beams = [([], 0.0)]
    for _ in range(max_len):
        candidates = []
        for seq, score in beams:
            if seq and seq[-1] == 0:
                candidates.append((seq, score)); continue
            logits = toy_next_logits(seq)
            logp = torch.log_softmax(logits, dim=-1)
            vals, idxs = torch.topk(logp, width)
            for v,i in zip(vals.tolist(), idxs.tolist()):
                candidates.append((seq+[i], score+v))
        beams = sorted(candidates, key=lambda x: x[1], reverse=True)[:width]
    return [(' '.join(vocab[i] for i in seq), score) for seq,score in beams]

beam_search()
